# 20. LEFT · RIGHT · FULL Outer Joins

*Module: 5 — Multi-table · Companion summary: `lectures/20-left-outer-join.md` · SQL engine: SQLite (in-memory)*

## What you will study

- keep unmatched rows from one or both sides
- run LEFT, RIGHT and FULL OUTER JOIN natively (SQLite ≥ 3.39)
- use COALESCE to label NULLs meaningfully
- read outer joins as data-quality detectors

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — borrower ⟕ loan, with *deliberate* gaps

- Peters/Li/Wong: borrowers with no loans (LEFT join will surface them as NULLs).
- One loan row points at borrower 'B999' who left the library — a data-quality issue
  the RIGHT join will surface. (No FK on this table, so the orphan can exist.)

In [ ]:
q("CREATE TABLE borrower (borrower_id TEXT PRIMARY KEY, lastname TEXT, country TEXT)")
q("INSERT INTO borrower VALUES ('BR1','Smith','CA'), ('BR2','Ali','IN'), ('BR3','Lee','CN'), ('BR4','Peters','US'), ('BR5','Li','CN'), ('BR6','Wong','CA')")
q("CREATE TABLE loan (borrower_id TEXT, copy_id TEXT, loan_date TEXT)")   # no FK: orphans allowed
q("""
INSERT INTO loan VALUES
    ('BR1','C1','2024-01-10'), ('BR1','C2','2024-02-01'),
    ('BR2','C3','2024-03-05'), ('BR3','C4','2024-04-01'),
    ('B999','C5','2024-05-01')
""")
q("SELECT * FROM borrower")
q("SELECT * FROM loan")

## 🧪 LEFT OUTER JOIN — all borrowers, matched or not

SQLite fills the no-match side with NULLs:

In [ ]:
q("""
SELECT b.lastname, l.copy_id, l.loan_date
FROM borrower b LEFT OUTER JOIN loan l ON b.borrower_id = l.borrower_id
""")

## 🧪 LEFT JOIN + WHERE col IS NULL — the anti-join

In [ ]:
q("""
SELECT b.lastname
FROM borrower b LEFT JOIN loan l ON b.borrower_id = l.borrower_id
WHERE l.borrower_id IS NULL
""")

## 🧪 RIGHT OUTER JOIN — all loans, orphaned borrower surfaced

In [ ]:
q("""
SELECT COALESCE(b.lastname, '❌ unknown borrower') AS who, l.copy_id, l.loan_date
FROM borrower b RIGHT OUTER JOIN loan l ON b.borrower_id = l.borrower_id
""")

## 🧪 FULL OUTER JOIN — nobody hides

Everything from both sides: matched pairs + unmatched borrowers + the orphan loan.

In [ ]:
q("""
SELECT COALESCE(b.lastname, '❌') AS borrower, COALESCE(l.copy_id, '—') AS copy, l.loan_date
FROM borrower b FULL OUTER JOIN loan l ON b.borrower_id = l.borrower_id
ORDER BY borrower, copy
""")

## 🧪 Diagram in words

| Join | Keeps everything from | Adds NULLs for |
|---|---|---|
| INNER | both, matched only | — |
| LEFT | left table | right-table misses |
| RIGHT | right table | left-table misses |
| FULL | both | all misses |

📕 Dialect note: MySQL has **no RIGHT/FULL** — emulate FULL as
`left-query UNION right-query` (UNION removes duplicates, which is exactly the FULL
join's semantics). SQL Server and SQLite ≥ 3.39 support all natively; this notebook
runs 3.49.

## 💡 Tips & tricks

- Orphan rows only appear with RIGHT/FULL joins — a great data-quality scan.
- LEFT JOIN + IS NULL = SQL's anti-join — memorize this pattern.
- Always COALESCE NULLs when presenting outer-join results.

## 🎯 Your turn

1. Loans (copy + date) for borrowers with no matching record — the orphan detector.
2. LEFT join version that shows every borrower, and the count of '' rows → how many
   borrowers have no loans?

In [ ]:
# 💡 Solution
q("SELECT l.copy_id, l.loan_date FROM borrower b RIGHT JOIN loan l ON b.borrower_id = l.borrower_id WHERE b.borrower_id IS NULL")
q("SELECT COUNT(*) AS borrowers_with_no_loans FROM (SELECT b.lastname FROM borrower b LEFT JOIN loan l ON b.borrower_id = l.borrower_id WHERE l.borrower_id IS NULL)")